# Pre-processing

In [ ]:
# OLD
# source = https://www.nordichq.com/guides/list-of-legal-entity-types-by-country-in-europe/
LEGAL_SUFFIXES = {
    "bv", "bv", "b.v.",
    "nv", "n.v", "n.v.",
    "sa", "s.a", "s.a.",
    "ag", "a.g", "a.g.",
    "vof", "v.o.f", "v.o.f.",
    "cv", "c.v", "c.v.",
    "eg", "e.g", "e.g.",
    "stichting", "maatschap",
    "vzw", "v.z.w", "v.z.w.", 
    "sl", "s.l", "s.l.", 
    "llp", "l.l.p", "l.l.p.",
    "ohg", "o.h.g", "e.v.", 
    "sarl", "s.a.r.l", 
    "sca", "s.c.a", "scs", "s.c.s",
    "gmbh", "llc", "l.l.c", "l.l.c.",
    "aps", "a/s", "ivs", "k/s", "f.m.b.a.",
    "ltd", "limited",
    "plc", "inc", "inc.", "corp", "corp.",
    "co", "kg", "k.g", "kgaa", "oy",
    "srl", "spa", "pte", "ab", "as"
}

# Improve Blocking

## run sklearn pipeline with svd

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=512,     # 256–1024 typical
    random_state=42
)

X_reduced = svd.fit_transform(G_tfidf)
# 450k x 260k ~ 3min runtime

In [ ]:
from sklearn.preprocessing import normalize

X_reduced = normalize(X_reduced, norm='l2', axis=1).astype('float32')

In [ ]:
S_reduced = svd.transform(S_tfidf)
S_reduced = normalize(S_reduced, norm='l2', axis=1).astype('float32')

In [ ]:
from sklearn.neighbors import NearestNeighbors

svdnn = NearestNeighbors(
    n_neighbors=10,
    metric='cosine',
    algorithm='auto',
    n_jobs=-1
)

svdnn.fit(X_reduced)

In [ ]:
# takes longrer than without svd (because it uses a dense array)
distances, indices = svdnn.kneighbors(S_reduced)

----

## TRY FAISS + SVD

In [ ]:
import faiss

dim = X_reduced.shape[1]

index = faiss.IndexHNSWFlat(dim, 32)

index.hnsw.efConstruction = 200
index.hnsw.efSearch = 64

index.add(X_reduced)

In [ ]:
distances, indices = index.search(
    S_reduced,  
    10          
)

## non-SVD FAISS

In [ ]:
import faiss

X = G_tfidf.astype('float32').toarray()

dim = X.shape[1]

index = faiss.IndexHNSWFlat(dim, 32)
index.hnsw.efConstruction = 200

index.add(X)

# FEATURE ENGINEERING
### TRY PHONETIC FEATURES

In [ ]:
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler
from phonetics import metaphone

def compute_features(row):
    """
    Compute a set of similarity and token-based features for a candidate pair of company names.
    Expects a row with 's_name_clean' and 'g_name_clean' columns.
    Returns a pandas Series with feature values.
    """
    # Extract cleaned names for S and G
    s = row["s_name_clean"]
    g = row["g_name_clean"]

    # Handle missing or non-string values gracefully
    if not isinstance(s, str):
        s = ""
    if not isinstance(g, str):
        g = ""

    # Tokenize both names into sets of words
    s_tokens = tokenize(s)
    g_tokens = tokenize(g)

    # Calculate token overlap (intersection and union)
    intersection = len(s_tokens & g_tokens)  # Number of shared tokens
    union = len(s_tokens | g_tokens)         # Total unique tokens across both names

    # Jaccard similarity: ratio of shared tokens to total unique tokens
    jaccard = intersection / union if union > 0 else 0

    # Length ratio: shorter name length divided by longer name length
    len_ratio = min(len(s), len(g)) / max(len(s), len(g)) if max(len(s), len(g)) > 0 else 0

    # Metaphone phonetic encoding for both names
    s_meta = metaphone(s)
    g_meta = metaphone(g)
    meta_jw = JaroWinkler.normalized_similarity(s_meta, g_meta)
    s_meta_tokens = {metaphone(tok) for tok in s_tokens if tok}
    g_meta_tokens = {metaphone(tok) for tok in g_tokens if tok}
    meta_intersection = len(s_meta_tokens & g_meta_tokens)
    meta_union = len(s_meta_tokens | g_meta_tokens)
    
    # Jaccard‑like phonetic similarity
    meta_token_jaccard = (
        meta_intersection / meta_union
        if meta_union > 0 else 0
    )
    # Compare best matching metaphone tokens (JW similarity for best pair)
    meta_token_sim = max(
    (
        JaroWinkler.normalized_similarity(a, b)
        for a in s_meta_tokens
        for b in g_meta_tokens
    ),
    default=0
)

    # Compute and return all features as a Series
    return pd.Series({
        # Fuzzy string similarity metrics (normalized to [0,1])
        # "lev_ratio": Levenshtein.normalized_similarity(s, g), # Normalized Levenshtein similarity
        "jaro_winkler": JaroWinkler.normalized_similarity(s, g), # Jaro-Winkler similarity
        "token_set_ratio": fuzz.token_set_ratio(s, g) / 100, # Token set ratio (compares shared words only)
        "token_sort_ratio": fuzz.token_sort_ratio(s, g) / 100, # Token sort ratio (sorts tokens alphabetically before comparing)
        "partial_ratio": fuzz.partial_ratio(s, g) / 100, # Partial ratio (looks for best matching substring)

        # Token-based features
        "jaccard": jaccard, # Jaccard similarity score

        # Length-based features
        "len_ratio": len_ratio, # Ratio of name lengths
        "len_diff": abs(len(s) - len(g)), # Absolute difference in name lengths

        # Phonetic features
        "metaphone_match": meta_jw # Jaro-Winkler similarity of metaphone codes
        "metaphone_jaccard": meta_token_jaccard, # Jaccard similarity of metaphone codes
        "metaphone_jw": meta_token_sim # Jaro-Winkler similarity of metaphone codes
    })

In [ ]:
feature_cols = [
    "tfidf_similarity",
    "jaro_winkler",
    "token_set_ratio",
    "token_sort_ratio",
    "partial_ratio",
    "jaccard",
    "len_ratio",
    "len_diff",
    "metaphone_jaccard",
    "metaphone_jw"
]

X = train_df[feature_cols]
y = train_df["match"]

# Model Training
### Custom cost for CV search

In [ ]:
def cv_cost(train_df, xgb_probs):

    cv_df = train_df.copy()
    cv_df["match_prob"] = xgb_probs

    best_candidates = (
        cv_df
        .sort_values("match_prob", ascending=False)
        .groupby("train_index")
        .first()
        .reset_index()
    )
    # Sweep thresholds to evaluate cost, FP, FN, TP at each value
    thresholds = np.linspace(0,1,200)

    results = []

    for t in thresholds:
        # Compute cost and error counts for each threshold
        cost, fp, fn, tp = compute_cost(best_candidates, t)
        results.append([t, cost, fp, fn, tp])

    cost_df = pd.DataFrame(
        results,
        columns=["threshold","cost","fp","fn","tp"]
    )

    min_cost = cost_df["cost"].argmin()

    return min_cost

In [ ]:
from sklearn.metrics import make_scorer

def business_cost_scorer(estimator, X, y_true):
    """
    Custom scorer for GridSearchCV/RandomizedSearchCV that computes business cost.
    Returns negative cost (since sklearn maximizes scores, but we want to minimize cost).
    """
    # Get predicted probabilities
    y_probs = estimator.predict_proba(X)[:, 1]
    
    # Get the indices from X to map back to train_df
    # This assumes X is a DataFrame with original indices preserved
    indices = X.index
    
    # Create a temporary dataframe with predictions for this fold
    fold_df = train_df.loc[indices].copy()
    fold_df["match_prob"] = y_probs
    
    # Select best candidate per train_index (highest probability)
    best_candidates = (
        fold_df
        .sort_values("match_prob", ascending=False)
        .groupby("train_index")
        .first()
        .reset_index()
    )
    
    # Find optimal threshold by sweeping
    thresholds = np.linspace(0, 1, 200)
    min_cost = float('inf')
    
    for t in thresholds:
        cost, _, _, _ = compute_cost(best_candidates, t)
        if cost < min_cost:
            min_cost = cost
    
    # Return negative cost (sklearn maximizes, we want to minimize)
    return -min_cost


# Create the scorer object
custom_scorer = make_scorer(
    business_cost_scorer,
    greater_is_better=True,  # We return negative cost, so higher (less negative) is better
    needs_proba=False  # We handle probabilities ourselves
)

# Use it in RandomizedSearchCV
search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=1, random_state=42),
    param_dist,
    n_iter=50,
    cv=3,
    scoring=custom_scorer,  # Use your custom scorer
    n_jobs=-1,
    verbose=2
)

search.fit(X, y)
print("Best params:", search.best_params_)
print("Best score (negative cost):", search.best_score_)

# THRESHOLD OPTIMIZATION
### ALT APPROACH

In [ ]:
best_candidates["is_true_match"] = (
    best_candidates["candidate_company_id"]
    == best_candidates["true_company_id"]
)

best_candidates["true_in_G"] = (
    best_candidates["true_company_id"] != -1
)

In [ ]:
def evaluate_threshold(df, t):

    # apply threshold decision rule
    df["predicted_company_id"] = np.where(
        df["match_prob"] > t,
        df["candidate_company_id"],
        -1
    )

    # ground truth
    true = df["true_company_id"]
    pred = df["predicted_company_id"]

    # False Positive:
    # predicted match but wrong G_id
    FP = ((pred != -1) & (pred != true)).sum()

    # False Negative:
    # predicted -1 but actually in G
    FN = ((pred == -1) & (true != -1)).sum()

    cost = 5*FP + FN

    return FP, FN, cost

In [ ]:
thresholds = np.linspace(0, 1, 200)

results = []

for t in thresholds:
    FP, FN, cost = evaluate_threshold(best_candidates.copy(), t)
    results.append([t, FP, FN, cost])

threshold_results = pd.DataFrame(
    results,
    columns=["threshold", "FP", "FN", "cost"]
)

In [ ]:
best_row = threshold_results.loc[
    threshold_results["cost"].idxmin()
]

t_star = best_row["threshold"]

print("Optimal threshold:", t_star)
print(best_row)

Optimal threshold: 0.7638190954773869
threshold        0.763819
FP            4285.000000
FN           43769.000000
cost         65194.000000
Name: 152, dtype: float64


In [ ]:
best_candidates["predicted_company_id"] = np.where(
    best_candidates["match_prob"] > t_star,
    best_candidates["candidate_company_id"],
    -1
)